In [1]:
import sklearn
import numpy as np

iris = sklearn.datasets.load_iris()

X = iris.data.astype(np.float64)
y = iris.target.astype(np.int64)

In [2]:
np.shape(X), np.shape(y)

((150, 4), (150,))

In [3]:
rng = np.random.default_rng(0)

train_idx = []
test_idx = []

for class_id in (0, 1, 2):
  idx = np.flatnonzero(y == class_id)
  rng.shuffle(idx)

  train_idx.extend(idx[:35])
  test_idx.extend(idx[35:])

X_train = X[train_idx]
X_test = X[test_idx]

y_train = y[train_idx]
y_test = y[test_idx]

np.bincount(y_train), np.bincount(y_test)

(array([35, 35, 35]), array([15, 15, 15]))

In [4]:
mean = X_train.mean(axis=0)
std = X_train.std(axis=0, ddof=0)

X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

In [5]:
rng = np.random.default_rng(0)

W1 = rng.normal(loc=0.0, scale=np.sqrt(2 / 4), size=(4, 8)) # 4x8
b1 = np.zeros(8, dtype=np.float64)

W2 = rng.normal(loc=0.0, scale=np.sqrt(2 / (8 + 3)), size=(8, 3)) # 8x3
b2 = np.zeros(3, dtype=np.float64)

In [6]:
def forward(x):
  z1 = x @ W1 + b1 # nx4 @ 4x8 = nx8
  a1 = np.maximum(z1, 0) # nx8
  z2 = a1 @ W2 + b2 # nx8 @ 8x3 = nx3

  return z1, a1, z2


def backward(X, y, z1, a1, probs, mean=True):
  dz2 = probs.copy() # n x 3
  dz2[np.arange(len(y)), y] -= 1

  if mean:
    dz2 /= len(y)

  dW2 = a1.T @ dz2          # 8xn @ nx3 = 8x3
  db2 = np.sum(dz2, axis=0) # 1x3 

  da1 = dz2 @ W2.T     # nx3 @ 3x8 = nx8
  dz1 = da1 * (z1 > 0) # nx8

  dW1 = X.T @ dz1           # 4xn @ nx8 = 4x8
  db1 = np.sum(dz1, axis=0)

  return dW2, db2, dW1, db1


def cross_entropy(logits, y):
  shifted = logits.copy()
  shifted = shifted - np.max(shifted, axis=1, keepdims=True)

  exp_logits = np.exp(shifted)
  exp_sum = np.sum(exp_logits, axis=1, keepdims=True)

  probs = exp_logits / exp_sum

  target_probs = probs[np.arange(len(y)), y]
  loss = -np.log(target_probs)

  return loss.mean(), probs


In [7]:
z1, a1, logits = forward(X_train)
loss, probs = cross_entropy(logits, y_train)

dW2, db2, dW1, db1 = backward(X_train, y_train, z1, a1, probs)

np.shape(dW2), np.shape(db2), np.shape(dW1), np.shape(db1)

((8, 3), (3,), (4, 8), (8,))

In [8]:
import torch
import torch.nn.functional as F

X_torch = torch.tensor(X_train, dtype=torch.float64)
y_torch = torch.tensor(y_train, dtype=torch.long)

layer1 = torch.nn.Linear(4, 8, dtype=torch.float64)
layer2 = torch.nn.Linear(8, 3, dtype=torch.float64)

with torch.no_grad():
  layer1.weight.copy_(torch.from_numpy(W1.T))
  layer1.bias.copy_(torch.from_numpy(b1))

  layer2.weight.copy_(torch.from_numpy(W2.T))
  layer2.bias.copy_(torch.from_numpy(b2))

logits_torch = layer2(torch.relu(layer1(X_torch)))
loss_torch = F.cross_entropy(logits_torch, y_torch)

loss_torch.backward()

In [9]:
torch_grads = {
  "W1": layer1.weight.grad.detach().numpy().T,
  "b1": layer1.bias.grad.detach().numpy(),
  "W2": layer2.weight.grad.detach().numpy().T,
  "b2": layer2.bias.grad.detach().numpy(),
}

numpy_grads = {
  "W1": dW1,
  "b1": db1,
  "W2": dW2,
  "b2": db2,
}

print("NumPy loss:", loss)
print("PyTorch loss:", loss_torch.item())

tol = 1e-12
loss_diff = abs(loss - loss_torch.item())
print("loss", loss_diff, loss_diff <= tol)

for name in ("W1", "b1", "W2", "b2"):
  diff = np.max(np.abs(torch_grads[name] - numpy_grads[name]))
  finite = np.all(np.isfinite(torch_grads[name])) and np.all(np.isfinite(numpy_grads[name]))
  print(name, diff, finite and diff <= tol)

NumPy loss: 1.4562007801142816
PyTorch loss: 1.4562007801142818
loss 2.220446049250313e-16 True
W1 2.7755575615628914e-17 True
b1 2.7755575615628914e-17 True
W2 5.551115123125783e-17 True
b2 1.1102230246251565e-16 True


In [10]:
eps = 1e-6

def loss_on_train():
  _, _, logits = forward(X_train)
  value, _ = cross_entropy(logits, y_train)
  return value

def numerical_partial(param, index):
  original = param[index].item()

  param[index] = original + eps
  loss_plus = loss_on_train()

  param[index] = original - eps
  loss_minus = loss_on_train()

  param[index] = original

  return (loss_plus - loss_minus) / (2 * eps)

manual = {
  "W1[0, 0]": dW1[0, 0],
  "b1[0]": db1[0],
  "W2[0, 0]": dW2[0, 0],
  "b2[0]": db2[0],
}

targets = {
  "W1[0, 0]": (W1, (0, 0)),
  "b1[0]": (b1, 0),
  "W2[0, 0]": (W2, (0, 0)),
  "b2[0]": (b2, 0),
}

for name, (param, index) in targets.items():
  g_num = numerical_partial(param, index)
  g_manual = manual[name]
  diff = abs(g_num - g_manual)
  
  print(name, float(g_manual), g_num, diff, diff <= 1e-7)

W1[0, 0] -0.01992018433055532 -0.019920184390898044 6.034272387323014e-11 True
b1[0] -0.07056611158437998 -0.0705661116207068 3.632683043264251e-11 True
W2[0, 0] 0.13523020860547605 0.13523020869765645 9.218040220027035e-11 True
b2[0] 0.08552164484195579 0.08552164498798476 1.460289666965764e-10 True


In [11]:
dW2_bug, db2_bug, dW1_bug, db1_bug = backward(
  X_train, y_train, z1, a1, probs, mean=False
)

bug_grads = {
  "W1": dW1_bug,
  "b1": db1_bug,
  "W2": dW2_bug,
  "b2": db2_bug,
}

for name in ("W1", "b1", "W2", "b2"):
  diff = np.max(np.abs(bug_grads[name] - torch_grads[name]))
  print("pytorch", name, diff, diff <= 1e-12)

bug_manual = {
  "W1[0, 0]": dW1_bug[0, 0],
  "b1[0]": db1_bug[0],
  "W2[0, 0]": dW2_bug[0, 0],
  "b2[0]": db2_bug[0],
}

for name, (param, index) in targets.items():
  g_num = numerical_partial(param, index)
  diff = abs(g_num - bug_manual[name])
  print("numerical", name, float(bug_manual[name]), g_num, diff, diff <= 1e-7)

pytorch W1 18.518137980759192 False
pytorch b1 15.641093948478263 False
pytorch W2 34.193432939632565 False
pytorch b2 11.197086181821389 False
numerical W1[0, 0] -2.091619354708308 -0.019920184390898044 2.07169917031741 False
numerical b1[0] -7.409441716359898 -0.0705661116207068 7.338875604739191 False
numerical W2[0, 0] 14.19917190357499 0.13523020869765645 14.063941694877334 False
numerical b2[0] 8.979772708405358 0.08552164498798476 8.894251063417373 False
